# Student Performance Tracker - Exploratory Data Analysis (EDA)

This notebook provides a detailed walkthrough of the exploratory data analysis phase for the Student Performance Tracker. We load the dataset, perform statistical summaries, check for missing values and duplicates, and plot several critical distributions and correlations.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure parent directory is in path to import src modules
sys.path.append(os.path.abspath('..'))
from src.utils import load_data, configure_plotting_style

## 1. Load Dataset & Overview

In [ ]:
# Load the prepared dataset
df = load_data('../dataset/student_performance.csv')
print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## 2. Missing Value Analysis

In [ ]:
# Analyze missing values per column
missing_counts = df.isnull().sum()
missing_percentage = (missing_counts / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing_counts,
    'Percentage (%)': missing_percentage.round(2)
})
print("Missing values summary:")
missing_df

## 3. Duplicate Checking

In [ ]:
# Check for duplicate rows in the dataset
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

## 4. Statistical Summary

In [ ]:
# Statistical overview of numeric columns
df.describe().T

In [ ]:
# Overview of categorical columns
df.describe(include=['object']).T

## 5. Exploratory Data Visualizations

In [ ]:
# Configure consistent premium plotting style
configure_plotting_style()
os.makedirs('plots', exist_ok=True)

### 5.1 Correlation Heatmap

In [ ]:
numeric_cols = ['Age', 'Study_Hours', 'Attendance', 'Previous_Score', 'Sleep_Hours', 'Final_Score']
plt.figure(figsize=(8, 6))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Correlation Heatmap')
plt.savefig('plots/correlation_heatmap.png')
plt.show()

### 5.2 Score Distribution

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df['Final_Score'].dropna(), kde=True, color='#4E79A7', bins=15)
plt.axvline(df['Final_Score'].mean(), color='red', linestyle='--', label=f"Mean: {df['Final_Score'].mean():.1f}")
plt.axvline(df['Final_Score'].median(), color='green', linestyle='-', label=f"Median: {df['Final_Score'].median():.1f}")
plt.title('Distribution of Final Exam Scores')
plt.xlabel('Final Score (%)')
plt.ylabel('Count')
plt.legend()
plt.savefig('plots/score_distribution.png')
plt.show()

### 5.3 Attendance Distribution

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df['Attendance'].dropna(), kde=True, color='#76B7B2', bins=15)
plt.axvline(df['Attendance'].mean(), color='red', linestyle='--', label=f"Mean: {df['Attendance'].mean():.1f}%")
plt.title('Distribution of Student Attendance')
plt.xlabel('Attendance Percentage (%)')
plt.ylabel('Count')
plt.legend()
plt.savefig('plots/attendance_distribution.png')
plt.show()

### 5.4 Study Hours vs. Final Score

In [ ]:
plt.figure(figsize=(8, 5))
sns.regplot(data=df.dropna(subset=['Study_Hours', 'Final_Score']), x='Study_Hours', y='Final_Score', 
            scatter_kws={'alpha':0.6, 'color':'#F28E2B'}, line_kws={'color':'red', 'linewidth':2})
plt.title('Weekly Study Hours vs. Final Exam Score')
plt.xlabel('Weekly Study Hours')
plt.ylabel('Final Score (%)')
plt.savefig('plots/study_hours_vs_score.png')
plt.show()

### 5.5 Feature Histograms

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    sns.histplot(df[col].dropna(), kde=True, ax=axes[i], color='#59A14F', bins=15)
    axes[i].set_title(f'Histogram of {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')
plt.suptitle('Histograms of Numerical Features', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('plots/histograms_numerical.png')
plt.show()

### 5.6 Boxplots (Final Score by Categorical Factors)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Gender boxplot
sns.boxplot(data=df.dropna(subset=['Gender', 'Final_Score']), x='Gender', y='Final_Score', ax=axes[0], palette='pastel')
axes[0].set_title('Final Score by Gender')

# Parent Education boxplot
edu_order = ['None', 'Primary Education', '5th to 9th Grade', 'Secondary Education', 'Higher Education']
edu_clean = df.dropna(subset=['Parent_Education', 'Final_Score'])
present_order = [lvl for lvl in edu_order if lvl in edu_clean['Parent_Education'].unique()]
sns.boxplot(data=edu_clean, x='Parent_Education', y='Final_Score', ax=axes[1], order=present_order, palette='Set2')
axes[1].set_title('Final Score by Parent Education')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30, ha='right')

# Family Support boxplot
sns.boxplot(data=df.dropna(subset=['Family_Support', 'Final_Score']), x='Family_Support', y='Final_Score', ax=axes[2], palette='Set3')
axes[2].set_title('Final Score by Family Support')

plt.suptitle('Boxplots: Final Score Analysis by Categorical Factors', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig('plots/boxplots_analysis.png')
plt.show()